[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/09-use-cases/01-finding_duplicate_customers.ipynb)

In [1]:
# !pip install mbox pandas

# Finding Duplicate Customers in a Table

A straightforward, common application: you have a customer table in a SQL database, and some fraction of the rows are actually the same person entered more than once, typo'd, nicknamed, just slightly different each time. The goal is a DataFrame with two extra columns for every record: `is_duplicate`, and `similar_candidates`, the other row IDs that look like the same person.

In this notebook you will:

1. Load a customer table straight out of a SQL database, 10,000 rows, 564 of them are real duplicates of another row
2. Match the table against itself, and build the `is_duplicate` / `similar_candidates` columns
3. Check the result against the known duplicates
4. Add an `AliasSet` to catch the nicknames plain matching misses, and see the improvement

## 1. The data

`datasets/customers.db` is a SQLite database with 10,000 customers, 564 of them are deliberate duplicates, a corrupted copy of another row already in the table, a nickname, a typo, or both. It also carries a second table, `duplicate_pairs`, the answer key for which rows those are, purely so this notebook can check its own work in Section 4, a real dataset wouldn't have that.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("datasets/customers.db")
customers_table = pd.read_sql_query("select * from customers", conn)
print(f"{len(customers_table)} rows loaded from the customers table")
customers_table.head()

10000 rows loaded from the customers table


,customer_id,full_name,email
0,1,Martha Reyes,martha.reyes@example.com
1,2,Joyce Castillo,joyce.castillo@example.com
2,3,Sean Gray,sean.gray@example.com
3,4,Ethan Myers,ethan.myers@example.com
4,5,Deborah Martin,deborah.martin@example.com


## 2. Match the table against itself

One index, built from the full table. Each row is then searched against that same index, `max_results=4` so every row gets its exact self-match plus up to three genuine candidates. The self-match, `index_row` equal to the row's own position, is dropped immediately, it isn't a duplicate of itself.

In [3]:
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

index = TableIndexer.create_index(customers_table, index_columns=["full_name", "email"], tmp_dir="tmp_index_customers")

config = TableRecallConfig(
    fields=[
        TableRecallFieldConfig(input_column="full_name", indexed_column="full_name", minimum_quality=0, weight=70, mode=TableRecallMode.APPROX),
        TableRecallFieldConfig(input_column="email", indexed_column="email", minimum_quality=0, weight=30, mode=TableRecallMode.APPROX),
    ],
    max_results=4, min_total_match_value=0, include_field_scores=True
)

raw_matches = index.match(queries=customers_table[["full_name", "email"]], config=config)
candidates = raw_matches[raw_matches["query_row"] != raw_matches["index_row"]]
print(f"{len(raw_matches)} candidate rows returned, {len(candidates)} after dropping each row's match on itself")

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


20799 candidate rows returned, 10799 after dropping each row's match on itself


`email` is included even though it's not the field with typos in it, on purpose: two genuinely different people can have similar-sounding names by pure chance at 10,000 rows (`"Thomas Scott"` and `"Scott Thomas"` are a real example in this dataset), but they won't share an email too. Weighting `full_name` higher but keeping `email` in the mix is what keeps those coincidences from being flagged.

## 3. Building `is_duplicate` and `similar_candidates`

A threshold turns a raw score into a decision. `90` is used here because it was checked directly against this dataset's actual score distribution, not guessed, real duplicates mostly score in the high 80s to 100, coincidental look-alikes mostly land well below that.

In [4]:
THRESHOLD = 90
flagged = candidates[candidates["overall_score"] >= THRESHOLD]

similar_by_row = flagged.groupby("query_row")["index_row"].apply(list).to_dict()

customers_table["similar_candidates"] = [
    [int(customers_table.iloc[j]["customer_id"]) for j in similar_by_row.get(i, [])]
    for i in range(len(customers_table))
]
customers_table["is_duplicate"] = customers_table["similar_candidates"].apply(len) > 0

print(f"{customers_table['is_duplicate'].sum()} rows flagged as having a duplicate elsewhere in the table")
customers_table[customers_table["is_duplicate"]].head()

660 rows flagged as having a duplicate elsewhere in the table


,customer_id,full_name,email,similar_candidates,is_duplicate
0,1,Martha Reyes,martha.reyes@example.com,[9437],True
51,52,Helen Morgan,helen.morgan@example.com,[9438],True
64,65,Nancy Jones,nancy.jones@example.com,[9439],True
124,125,Adam Hill,adam.hill@example.com,[9441],True
178,179,Evelyn Rodriguez,evelyn.rodriguez@example.com,[9443],True


That's the deliverable, `customers_table` now carries `is_duplicate` and `similar_candidates` for every one of the 10,000 rows, ready to hand to whatever merges or reviews duplicates downstream.

## 4. Checking the result

This dataset's answer key makes it possible to check the work directly, something a real dataset won't offer, but it's worth doing once here to know what this approach actually catches.

In [5]:
duplicate_pairs = pd.read_sql_query("select * from duplicate_pairs", conn)
truly_duplicated_ids = set(duplicate_pairs["duplicate_id"]) | set(duplicate_pairs["original_id"])

flagged_ids = set(customers_table.loc[customers_table["is_duplicate"], "customer_id"])
found = flagged_ids & truly_duplicated_ids
missed = truly_duplicated_ids - flagged_ids
false_positives = flagged_ids - truly_duplicated_ids

print(f"{len(found)} of {len(truly_duplicated_ids)} truly-duplicated rows correctly flagged")
print(f"{len(missed)} missed, {len(false_positives)} rows flagged that weren't actually duplicates")

640 of 1128 truly-duplicated rows correctly flagged
488 missed, 20 rows flagged that weren't actually duplicates


Not perfect, and that's expected, some of the 564 injected duplicates only changed a first name to a nickname, `"Bob Smith"` for `"Robert Smith"`, which plain `APPROX` fuzzy matching has very little to work with: the two words barely share a letter.

## 5. Catching nicknames with `AliasSet`

`02-data-harmonization/02` covers this in depth, a nickname isn't a spelling variation `APPROX` can fuzzy-match its way to, it's a different word for the same thing, and needs to be taught explicitly. `datasets/nicknames.csv` is a small, plain table of common English nicknames, `word`, `alias`, `penalty`, the same three columns `AliasSet.from_dataframe` expects.

In [6]:
from mbox.aliases import AliasSet

nicknames = pd.read_csv("datasets/nicknames.csv")
nickname_aliases = AliasSet.from_dataframe(df=nicknames, name="given_names")

aliased_index = TableIndexer.create_index(
    customers_table, index_columns=["full_name", "email"],
    alias_sets={"full_name": nickname_aliases}, tmp_dir="tmp_index_customers_aliased"
)

aliased_matches = aliased_index.match(queries=customers_table[["full_name", "email"]], config=config)
aliased_candidates = aliased_matches[aliased_matches["query_row"] != aliased_matches["index_row"]]
aliased_flagged = aliased_candidates[aliased_candidates["overall_score"] >= THRESHOLD]

aliased_similar_by_row = aliased_flagged.groupby("query_row")["index_row"].apply(list).to_dict()
customers_table["similar_candidates"] = [
    [int(customers_table.iloc[j]["customer_id"]) for j in aliased_similar_by_row.get(i, [])]
    for i in range(len(customers_table))
]
customers_table["is_duplicate"] = customers_table["similar_candidates"].apply(len) > 0

flagged_ids = set(customers_table.loc[customers_table["is_duplicate"], "customer_id"])
found = flagged_ids & truly_duplicated_ids
false_positives = flagged_ids - truly_duplicated_ids
print(f"{len(found)} of {len(truly_duplicated_ids)} truly-duplicated rows correctly flagged, "
      f"{len(false_positives)} false positives")

675 of 1128 truly-duplicated rows correctly flagged, 20 false positives


More real duplicates found, the same number of false positives, `AliasSet` only ever helps `full_name` recognize a nickname it already knows about, it has no way to manufacture a false match out of nothing. The rows still missed at this point are typically the ones where both a nickname *and* a heavy typo landed on the same record, genuinely the hardest case, and a candidate for a lower threshold reviewed by a human rather than auto-merged.

## 6. The final table

Every one of the 10,000 rows, `is_duplicate` and `similar_candidates` included, exactly as requested at the top of this notebook.

In [7]:
customers_table[customers_table["is_duplicate"]][["customer_id", "full_name", "email", "similar_candidates"]].head(10)

,customer_id,full_name,email,similar_candidates
0,1,Martha Reyes,martha.reyes@example.com,[9437]
51,52,Helen Morgan,helen.morgan@example.com,[9438]
64,65,Nancy Jones,nancy.jones@example.com,[9439]
124,125,Adam Hill,adam.hill@example.com,[9441]
178,179,Evelyn Rodriguez,evelyn.rodriguez@example.com,[9443]
192,193,Judith Bennett,judith.bennett@example.com,[9444]
219,220,Henry Murphy,henry.murphy@example.com,[9446]
230,231,Jack Bailey,jack.bailey@example.com,[9448]
261,262,Joshua Baker,joshua.baker@example.com,[9449]
274,275,Walter Cruz,walter.cruz@example.com,[9451]


## Practical notes

**Self-matching means dropping the row's match on itself.** `index_row == query_row` is guaranteed to be every row's top result, score `100`, since it's comparing a record to an identical copy of itself sitting in the same index. Forgetting to filter it out makes every single row look like a duplicate of itself.

**A second field earns its keep even when it's not where the noise is.** Section 2's `email` field wasn't corrupted in this dataset at all, but including it is what separates a real duplicate from two different people who just happen to have similar-sounding names, a real, if rare, occurrence once a table has thousands of rows drawn from a small pool of common names.

**Pick the threshold from the data, not a guess.** `90` in Section 3 came from actually looking at where true duplicates and coincidental look-alikes land in this dataset. A different dataset, a different name distribution, will have a different real number.

**`AliasSet` raises recall, not false positives.** Section 5's improvement came entirely from finding more of the *real* duplicates, the false-positive count didn't move, because an alias only ever helps the engine recognize a word it's told about, it can't invent a match where nothing else in the record supports one.

## Next steps

- **`02-data-harmonization/02-alias_sets_nicknames_and_synonyms.ipynb`** - `AliasSet` in depth, including penalties, corporate synonyms, and reusable alias files
- **`05-explainability/07-multi_field_weights_explained.ipynb`** - more on why a second field changes which candidates are considered, not just how they're ranked